In [1]:
!pip install wikipedia

In [2]:
import wikipedia
import re

wikipedia.set_lang("en")

def fetch_wikipedia_page(title):
    try:
        # 🔥 先搜索
        results = wikipedia.search(title)
        
        if not results:
            print(f"❌ No results for {title}")
            return None
        
        # 🔥 取第一个最相关的
        page_title = results[0]
        
        page = wikipedia.page(page_title, auto_suggest=False)
        
        text = page.content
        
        # 清洗
        text = re.sub(r"\[\d+\]", "", text)
        text = re.sub(r"\n{2,}", "\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        
        return {
            "title": page.title,
            "text": text.strip()
        }
    
    except Exception as e:
        print(f"❌ Error fetching {title}: {e}")
        return None

In [3]:
topics = [
    "Japanese cuisine",
    "Chinese cuisine",
    "Korean cuisine",
    "Sushi",
    "Kimchi",
    "Ramen",
    "Soy sauce",
    "Tofu",
    "Tempura",
    "Udon",
    "Soba",
    "Peking duck",
    "Hot pot",
    "Dim sum",
    "Bibimbap",
    "Bulgogi",
    "Tteokbokki",
    "Rice",
    "Noodle",
    "Miso",
    "Seaweed",
    "Fish sauce",
    "Ginger",
    "Garlic",
    "Stir frying",
    "Steaming",
    "Deep frying",
    "Fermentation",
    "Braising",
    "Japanese tea ceremony",
    "Korean barbecue",
    "Street food",
]

corpus = []

for i, topic in enumerate(topics):
    print(f"Fetching: {topic}")
    
    data = fetch_wikipedia_page(topic)
    
    if data:
        corpus.append({
            "doc_id": f"doc_{i}",
            "title": data["title"],
            "text": data["text"]
        })

Fetching: Japanese cuisine
Fetching: Chinese cuisine
Fetching: Korean cuisine
Fetching: Sushi
Fetching: Kimchi
Fetching: Ramen
Fetching: Soy sauce
Fetching: Tofu
Fetching: Tempura
Fetching: Udon
Fetching: Soba
Fetching: Peking duck
Fetching: Hot pot
Fetching: Dim sum
Fetching: Bibimbap
Fetching: Bulgogi
Fetching: Tteokbokki
Fetching: Rice
Fetching: Noodle
Fetching: Miso
Fetching: Seaweed
Fetching: Fish sauce
Fetching: Ginger
Fetching: Garlic
Fetching: Stir frying
Fetching: Steaming
Fetching: Deep frying
Fetching: Fermentation
Fetching: Braising
Fetching: Japanese tea ceremony
Fetching: Korean barbecue
Fetching: Street food


In [35]:
import json

with open("corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, indent=4, ensure_ascii=False)

print("✅ corpus.json saved")

✅ corpus.json saved


start chunking!!!

In [36]:
import re
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/macbookpro/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [37]:
def clean_text(text):
    text = re.sub(r"\[\d+\]", "", text)
    text = text.replace("\r", "\n")
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

In [38]:
def split_into_sections(text):
    pattern = r"(==+.*?==+)"
    parts = re.split(pattern, text)

    sections = []
    current_title = "Introduction"
    current_text = ""

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if re.match(r"==+.*?==+", part):
            if current_text.strip():
                sections.append((current_title, current_text.strip()))
            current_title = part
            current_text = ""
        else:
            current_text += " " + part

    if current_text.strip():
        sections.append((current_title, current_text.strip()))

    return sections

In [39]:
def hybrid_chunk(text, max_length=500, overlap_sentences=1):
    sections = split_into_sections(text)
    chunks = []

    for section_title, section_text in sections:
        sentences = sent_tokenize(section_text)

        i = 0
        while i < len(sentences):
            current_chunk = section_title + " "
            j = i

            while j < len(sentences) and len(current_chunk) + len(sentences[j]) <= max_length:
                current_chunk += " " + sentences[j]
                j += 1

            chunks.append(current_chunk.strip())

            if j == i:
                i += 1
            else:
                i = max(j - overlap_sentences, i + 1)

    return chunks

In [40]:
all_chunks = []
chunk_id = 0

for doc in corpus:
    text = clean_text(doc["text"])
    
    chunks = hybrid_chunk(text)
    
    for c in chunks:
        if len(c.strip()) < 10:
            continue
        
        all_chunks.append({
            "chunk_id": f"chunk_{chunk_id}",
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "text": c.strip()
        })
        
        chunk_id += 1

In [41]:
import json

with open("chunks_hybrid.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=4, ensure_ascii=False)

print(f"✅ Total chunks: {len(all_chunks)}")

✅ Total chunks: 3031


start embedding!!!

In [42]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Model loaded")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 13275.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded


把JSON 里的text提取出来

In [43]:
chunk_texts = [c["text"] for c in all_chunks]

print(f"Total chunks: {len(chunk_texts)}")

Total chunks: 3031


编码，把每一条chunking 变成 向量

In [44]:
import numpy as np

print("🔄 Encoding chunks...")

chunk_embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=32
)
chunk_embeddings = chunk_embeddings / np.linalg.norm(
    chunk_embeddings, axis=1, keepdims=True
)
print("✅ Encoding done")
print(chunk_embeddings.shape)

🔄 Encoding chunks...


Batches: 100%|██████████████████████████████████| 95/95 [00:03<00:00, 30.45it/s]

✅ Encoding done
(3031, 384)


In [45]:
np.save("chunk_embeddings.npy", chunk_embeddings)

print("💾 Saved chunk embeddings")
print(chunk_embeddings.shape)

💾 Saved chunk embeddings
(3031, 384)


Retrieval 开始进行排序

把query变成 embedding，然后计算 query 和 每个chunk的 相似度。找最大 最相似的。

In [46]:
def retrieve(query, k=5):
    query_emb = model.encode(query, convert_to_numpy=True)
    query_emb = query_emb / np.linalg.norm(query_emb)

    scores = np.dot(chunk_embeddings, query_emb)
    top_indices = np.argsort(scores)[-k:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": all_chunks[idx]["chunk_id"],
            "doc_id": all_chunks[idx]["doc_id"],
            "title": all_chunks[idx]["title"],
            "text": all_chunks[idx]["text"],
            "score": float(scores[idx])
        })
    return results

测试

In [47]:
query = "What is sushi made of?"

results = retrieve(query)

for i, r in enumerate(results):
    print(f"\n[{i+1}] Score: {r['score']:.4f}")
    print(r["text"][:200])


[1] Score: 0.7615
Introduction  Sushi (すし, 寿司, 鮨, 鮓; pronounced [sɯɕiꜜ] or [sɯꜜɕi] ) is a traditional Japanese dish made with vinegared rice (鮨飯, sushi-meshi), typically seasoned with sugar and salt, and combined with 

[2] Score: 0.7459
== Ingredients ==  Traditional Japanese sushi consists of rice flavored with vinegar sauce and various raw or cooked ingredients.

[3] Score: 0.7370
Introduction  It was the fast food of the chōnin class in the Edo period. Sushi is traditionally made with medium-grain white rice, although it can also be prepared with brown rice or short-grain rice

[4] Score: 0.7236
== Nutrition ==  The main ingredients of traditional Japanese sushi—raw fish and rice—are naturally low in fat and high in protein, carbohydrates (from the rice), vitamins, and minerals, as are gari (

[5] Score: 0.7084
== Ingredients ==  All sushi has a base of specially prepared rice, complemented with other ingredients. Traditional Japanese sushi consists of rice flavored with vinegar s

加载Qwen模型

In [48]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
).to("cpu")   # 🔥 Mac建议用CPU

print("✅ Qwen loaded")

Loading weights: 100%|██████████████████████| 290/290 [00:00<00:00, 2831.15it/s]


✅ Qwen loaded


build_prompt 把chunk + query拼成一个输入

In [49]:
def build_prompt(query, contexts):
    context_text = "\n\n".join([c["text"] for c in contexts])

    prompt = f"""Answer the question using ONLY the context below.

Give a short answer copied from the context when possible.
Do not explain.
Do not continue the conversation.
If the answer is not clearly in the context, say only: I don't know

Context:
{context_text}

Question: {query}
Answer:"""

    return prompt

tokenizer: 把文字变成模型能读懂的格式
generate: LLM开始思考
decode: 把模型输出变回文字

In [70]:
import re

def postprocess_answer(answer):
    answer = answer.strip()
    answer = re.sub(r"^Introduction\s+", "", answer)
    answer = re.sub(r"^==+\s*[^=]+?\s*==+\s*", "", answer)
    answer = re.sub(r"^=+\s*[^=]+?\s*=+\s*", "", answer)
    answer = re.sub(r"\s+", " ", answer).strip()
    answer = answer.strip(" .,:;")
    return answer

In [71]:
import numpy as np

def extract_best_sentence(query, contexts, embed_model):
    all_sentences = []

    for c in contexts:
        for s in sent_tokenize(c["text"]):
            s = s.strip()
            if s:
                all_sentences.append(s)

    if not all_sentences:
        return "I don't know"

    sent_embs = embed_model.encode(all_sentences, convert_to_numpy=True)
    sent_embs = sent_embs / np.linalg.norm(sent_embs, axis=1, keepdims=True)

    query_emb = embed_model.encode(query, convert_to_numpy=True)
    query_emb = query_emb / np.linalg.norm(query_emb)

    scores = np.dot(sent_embs, query_emb)
    best_idx = int(np.argmax(scores))

    return all_sentences[best_idx]

In [81]:
import re
import numpy as np
from nltk.tokenize import sent_tokenize

def generate_answer(query):
    contexts = retrieve(query, k=5)

    if not contexts:
        return "I don't know"

    q = query.lower().strip()

    def normalize_text(s):
        s = postprocess_answer(s)
        s = re.sub(r"\s+", " ", s).strip()
        return s

    def get_all_candidate_sentences():
        candidates = []
        for c_idx, c in enumerate(contexts):
            for s_idx, s in enumerate(sent_tokenize(c["text"])):
                s_clean = normalize_text(s)
                if len(s_clean) >= 20:
                    candidates.append({
                        "text": s_clean,
                        "chunk_rank": c_idx,
                        "sent_rank": s_idx
                    })
        return candidates

    def score_candidates(candidates, strong_patterns, weak_patterns=None, avoid_patterns=None):
        if weak_patterns is None:
            weak_patterns = []
        if avoid_patterns is None:
            avoid_patterns = []

        scored = []
        for item in candidates:
            s = item["text"]
            s_lower = s.lower()
            score = 0.0

            for p in strong_patterns:
                if p in s_lower:
                    score += 6

            for p in weak_patterns:
                if p in s_lower:
                    score += 2

            for p in avoid_patterns:
                if p in s_lower:
                    score -= 8

            score += max(0, 5 - item["chunk_rank"]) * 1.2
            score += max(0, 4 - item["sent_rank"]) * 0.6
            score -= max(0, len(s) - 180) * 0.01

            scored.append((score, s))

        scored.sort(key=lambda x: x[0], reverse=True)
        return scored

    candidates = get_all_candidate_sentences()

    # WHERE
    if "come from" in q:
        strong_patterns = [
            "originated in",
            "originated from",
            "comes from",
            "come from"
        ]
        weak_patterns = [
            "is from",
            "are from",
            "from beijing",
            "portuguese jesuits"
        ]
        avoid_patterns = [
            "history",
            "etymology",
            "variation of"
        ]

        scored = score_candidates(candidates, strong_patterns, weak_patterns, avoid_patterns)
        if scored and scored[0][0] > 0:
            return scored[0][1]

    # HOW SERVED
    if "usually served" in q and "when" not in q:
        strong_patterns = [
            "traditionally served inside",
            "served inside",
            "traditionally carved in front of the guests and served",
            "served in three stages",
            "served with",
            "served on"
        ]
        weak_patterns = [
            "served in",
            "eaten with",
            "traditionally served"
        ]
        avoid_patterns = [
            "without rice or noodles",
            "daily routine",
            "history",
            "variation of",
            "restaurant",
            "covid"
        ]

        scored = score_candidates(candidates, strong_patterns, weak_patterns, avoid_patterns)
        if scored and scored[0][0] > 0:
            return scored[0][1]

    # WHEN
    if "when is" in q:
        strong_patterns = [
            "for brunch",
            "in the summer and hot in the winter",
            "chilled in the summer and hot in the winter",
            "at nearly every meal"
        ]
        weak_patterns = [
            "traditionally enjoyed",
            "early, around 10:00 am",
            "usually served",
            "usually eaten",
            "commonly served",
            "commonly eaten"
        ]
        avoid_patterns = [
            "daily routine",
            "way of life",
            "usually mixed",
            "stuffed eggplant",
            "history",
            "variation of"
        ]

        scored = score_candidates(candidates, strong_patterns, weak_patterns, avoid_patterns)
        if scored and scored[0][0] > 0:
            return scored[0][1]

    # USED FOR
    if "used for" in q:
        strong_patterns = [
            "used for",
            "is used for",
            "used as",
            "is used as"
        ]
        weak_patterns = [
            "used in",
            "is used in",
            "commonly used as",
            "commonly used in",
            "most commonly appears as",
            "appears as"
        ]
        avoid_patterns = [
            "history",
            "roman times",
            "etymology",
            "variation of"
        ]

        scored = score_candidates(candidates, strong_patterns, weak_patterns, avoid_patterns)
        if scored and scored[0][0] > 0:
            return scored[0][1]

    return postprocess_answer(extract_best_sentence(query, contexts, model))

测试

In [82]:
query = "What is sushi made of?"

answer = generate_answer(query)

print("Q:", query)
print("A:", answer)

Q: What is sushi made of?
A: Sushi (すし, 寿司, 鮨, 鮓; pronounced [sɯɕiꜜ] or [sɯꜜɕi] ) is a traditional Japanese dish made with vinegared rice (鮨飯, sushi-meshi), typically seasoned with sugar and salt, and combined with a variety of ingredients (ねた, neta), such as seafood, vegetables, or meat; raw seafood is the most common, although some may be cooked


evaluation！！

In [83]:
import json

with open("benchmark.json", "r", encoding="utf-8") as f:
    benchmark = json.load(f)

print(f"✅ Loaded {len(benchmark)} benchmark queries")

✅ Loaded 8 benchmark queries


In [84]:
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [85]:
results = []

for item in benchmark:
    query = item["query"]
    
    answer = generate_answer(query)   # ✅ 在这里调用
    
    results.append({
        "query": query,
        "answer": answer
    })

完全一样 = 1
否则 = 0

In [86]:
def exact_match(pred, gold):
    return int(normalize(pred) == normalize(gold))

In [87]:
from collections import Counter

def f1_score(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

In [88]:
for item in benchmark[:20]:
    query = item["query"]
    gold = item["answer"]
    pred = generate_answer(query)

    print("=" * 80)
    print("Q:", query)
    print("GOLD:", gold)
    print("PRED:", pred)
    print("EM:", exact_match(pred, gold))
    print("F1:", f1_score(pred, gold))

    retrieved = retrieve(query, k=3)
    for i, r in enumerate(retrieved):
        print(f"\n[Chunk {i+1}] score={r['score']:.4f}")
        print(r["text"][:300])

Q: What is soy sauce used for?
GOLD: Soy sauce may be added directly to food and is commonly used as a dipping sauce or used as seasoning in cooking.
PRED: Soy sauce may be added directly to food and is commonly used as a dipping sauce or used as seasoning in cooking
EM: 1
F1: 1.0

[Chunk 1] score=0.8032
== Use and storage ==  Soy sauce may be added directly to food and is commonly used as a dipping sauce or used as seasoning in cooking. It is often eaten with rice, noodles, sushi, or sashimi, or mixed with ground wasabi for dipping. Bottles of soy sauce for the purpose of seasoning dishes are commo

[Chunk 2] score=0.7690
Introduction  Soy sauce (sometimes called soya sauce in British English) is a liquid condiment of Chinese origin, traditionally made from a fermented paste of soybeans, roasted grain, brine, and Aspergillus oryzae or Aspergillus sojae molds. It is recognized for its saltiness and pronounced umami ta

[Chunk 3] score=0.7310
==== Blended ====  This soy sauce is made th

In [89]:
from collections import Counter
import numpy as np

em_scores = []
f1_scores = []

for item, pred in zip(benchmark, results):
    gold = item["answer"]
    pred_answer = pred["answer"]
    
    # EM
    em = exact_match(pred_answer, gold)
    em_scores.append(em)
    
    # F1
    f1 = f1_score(pred_answer, gold)
    f1_scores.append(f1)

# 最终结果
print("\n📊 FINAL RESULTS")
print(f"EM: {np.mean(em_scores):.4f}")
print(f"F1: {np.mean(f1_scores):.4f}")


📊 FINAL RESULTS
EM: 0.6250
F1: 0.6869
